# BiXRec v2 — full experiment run (Colab, GPU)

Runtime → Change runtime type → **GPU**. Runs are saved to Google Drive, so if the session stops you can
re-run all cells and finished runs are skipped.

Order: core (5 seeds × 3 datasets) → ablation → audit. Expect several hours in total; you can stop after
`core` and still build the main table.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/bixrec'   # results persist here
!mkdir -p $WORK

In [ ]:
%cd /content
!rm -rf paper && git clone -b v2 https://github.com/ASSIAbouamir/paper.git
%cd /content/paper
!pip -q install sentence-transformers scipy
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Data

In [ ]:
!python -m bixrec.download --root $WORK/raw --datasets ml1m beauty sports
!python -m bixrec.data ml1m   --raw $WORK/raw/ml-1m --out $WORK/data/ml1m
!python -m bixrec.data amazon --raw $WORK/raw/reviews_Beauty_5.json.gz --meta $WORK/raw/meta_Beauty.json.gz --out $WORK/data/beauty
!python -m bixrec.data amazon --raw $WORK/raw/reviews_Sports_and_Outdoors_5.json.gz --meta $WORK/raw/meta_Sports_and_Outdoors.json.gz --out $WORK/data/sports

In [ ]:
for ds in ['ml1m', 'beauty', 'sports']:
    !python -m bixrec.embed --data $WORK/data/{ds}

## 2. Quick check (one seed, ML-1M) — look at the numbers before launching the full grid

In [ ]:
!python -m bixrec.train --data $WORK/data/ml1m --variant sasrec --seed 42 --out $WORK/runs
!python -m bixrec.train --data $WORK/data/ml1m --variant bix --seed 42 --out $WORK/runs

## 3. Full grid (resumable)

In [ ]:
!python -m bixrec.run_all --data_root $WORK/data --runs $WORK/runs --stage core

In [ ]:
!python -m bixrec.run_all --data_root $WORK/data --runs $WORK/runs --stage ablation audit

## 4. Tables for the paper

In [ ]:
!python -m bixrec.make_tables --runs $WORK/runs --data_root $WORK/data --out $WORK/generated
!cp $WORK/generated/tables.tex paper/generated/tables.tex
!cd $WORK && zip -qr /content/bixrec_results.zip generated runs -x '*.pt'
from google.colab import files; files.download('/content/bixrec_results.zip')